# Bloque 7 — Un modelo sencillo de alcance
---
## Proyecto: Red Social Ficticia

Entrenamos un modelo que predice si una publicación tendrá **alto alcance** (12 000 visualizaciones o más) a partir de sus interacciones (likes, comentarios, compartidos…) y de los datos de su autor.

```text
datos validados → variables y objetivo → entrenamiento / prueba
   → candidatos (parámetros + métricas) → elección → modelo guardado en modelos/
```

Al final se guardan dos ficheros que **sí se suben al repositorio**:

| Fichero | Contenido |
|---|---|
| `modelos/modelo_alcance.joblib` | El modelo entrenado |
| `modelos/metadatos_modelo.json` | Tipo de modelo, **parámetros**, variables, semilla y **métricas** de prueba |

Las funciones están en `scripts/modelo.py` y los tests que comprueban el modelo guardado, en `tests/modelo/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import cross_val_score

# Raíz del proyecto: la primera carpeta (hacia arriba) que contiene pyproject.toml
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "scripts"))

import modelo as m
from contratos import construir_dataset_validado

## 1. Datos: variables y objetivo

Partimos del dataset procesado **validado con los contratos de datos**: si los datos de entrada no fueran válidos, el cuaderno se detendría aquí.

**Cuidado con la fuga de datos.** `visualizaciones` y `engagement_rate` no pueden ser variables: la primera es la propia respuesta y la segunda se calcula con ella. Un modelo que las usara parecería excelente, pero no serviría para nada (al publicar todavía no se conocen las visualizaciones).

In [ ]:
dataset = construir_dataset_validado(RAIZ / "data")
X, y = m.preparar_datos_modelo(dataset)

assert not set(m.VARIABLES) & set(m.VARIABLES_PROHIBIDAS), "¡Fuga de datos!"

print("Publicaciones:", len(X), "| variables:", m.VARIABLES)
print("Proporción de alto alcance:", round(y.mean(), 3))
X.head()

In [ ]:
X_train, X_test, y_train, y_test = m.dividir_datos(X, y)

print("Entrenamiento:", X_train.shape, "| prueba:", X_test.shape)
print("Alto alcance en train:", round(y_train.mean(), 3), "| en test:", round(y_test.mean(), 3))

## 2. Candidatos: parámetros y métricas

| Candidato | Para qué |
|---|---|
| `base` | Modelo trivial que predice al azar. Es la **referencia**: un modelo útil debe superarlo claramente |
| `logistica` | Regresión logística: sencilla e interpretable |
| `bosque` | Bosque aleatorio: capta relaciones no lineales |

Para **elegir** usamos la validación cruzada sobre el conjunto de entrenamiento (`cv_roc_auc`). El conjunto de prueba solo sirve para medir el modelo elegido al final, sin haberlo usado para decidir.

| Métrica | Qué mide |
|---|---|
| `accuracy` | Proporción de aciertos |
| `precision` | De las que el modelo dice "alto alcance", cuántas lo son |
| `recall` | De las que tienen alto alcance, cuántas detecta el modelo |
| `f1` | Equilibrio entre precision y recall |
| `roc_auc` | Capacidad de ordenar las publicaciones de más a menos probable (0.5 = azar, 1 = perfecto) |

In [ ]:
CANDIDATOS = {
    "base": {},
    "logistica": {"C": 1.0},
    "bosque": {"n_estimators": 200, "max_depth": 5},
}

modelos = {}
filas = []
for nombre, parametros in CANDIDATOS.items():
    modelo = m.crear_modelo(nombre, **parametros)
    cv_roc_auc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="roc_auc").mean()
    modelo.fit(X_train, y_train)
    modelos[nombre] = modelo
    filas.append({"modelo": nombre, "parametros": parametros, "cv_roc_auc": round(cv_roc_auc, 4),
                  **m.evaluar(modelo, X_test, y_test)})

comparacion = pd.DataFrame(filas).set_index("modelo").sort_values("cv_roc_auc", ascending=False)
comparacion

## 3. Elección del modelo

Elegimos el candidato con mejor `cv_roc_auc` (excluyendo la referencia) y comprobamos que supera claramente al modelo trivial.

In [ ]:
elegido = comparacion.drop(index="base")["cv_roc_auc"].idxmax()
metricas_test = m.evaluar(modelos[elegido], X_test, y_test)

print("Modelo elegido:", elegido, "| parámetros:", CANDIDATOS[elegido])
print("Métricas de prueba:", metricas_test)
print("Mejora sobre la referencia (roc_auc):", round(metricas_test["roc_auc"] - comparacion.loc["base", "roc_auc"], 3))

## 4. Guardar el modelo y sus metadatos

Guardamos el modelo con `joblib` y, a su lado, un JSON con sus **parámetros** y **métricas**. Así cualquiera (y los tests) puede saber qué modelo es, cómo se entrenó y cómo de bueno es.

In [ ]:
metadatos = m.guardar_modelo(modelos[elegido], elegido, CANDIDATOS[elegido], metricas_test)
print("Guardado en:", m.RUTA_MODELO.relative_to(RAIZ), "y", m.RUTA_METADATOS.relative_to(RAIZ))
metadatos

In [ ]:
# Comprobación: el modelo guardado se carga y predice
modelo_guardado = m.cargar_modelo()
ejemplos = X_test.head(5)
pd.DataFrame({
    "probabilidad_alto_alcance": modelo_guardado.predict_proba(ejemplos)[:, 1].round(3),
    "prediccion": modelo_guardado.predict(ejemplos),
    "real": y_test.head(5).to_numpy(),
}, index=ejemplos.index)

## 5. Para practicar

1. Añade a `CANDIDATOS` otra configuración (por ejemplo, `"logistica": {"C": 0.1}` o un bosque con más árboles) y vuelve a ejecutar el cuaderno. ¿Cambia el modelo elegido? ¿Y `metadatos_modelo.json`?
2. Ejecuta `uv run pytest -m modelo`. ¿Qué comprueba cada test?
3. Guarda a propósito el modelo `base` y vuelve a lanzar los tests: ¿cuáles fallan y por qué?
4. Haz commit del modelo en tu rama `feature/` y observa en GitHub Actions el workflow del modelo.